# Quy trình tiền xử lý dữ liệu bình luận mạng xã hội

Notebook này trình bày quy trình khảo sát và tiền xử lý có thể chạy lại trên
snapshot `Preprocessing/data/raw/raw.csv` của bộ dữ liệu bình luận mạng xã hội.

Các mục từ 1 đến 5.4 đã được triển khai bằng cell code và giữ nguyên dữ liệu
raw để có thể đối chiếu. Phần khử trùng lặp được xử lý ở notebook riêng.

## 1. Mục tiêu và phạm vi

Mục tiêu của quy trình là đưa dữ liệu thu thập từ các nền tảng khác nhau về
cấu trúc thống nhất, phát hiện dữ liệu lỗi và tạo đầu vào sạch hơn, đồng thời
bảo toàn cách biểu đạt tự nhiên của người dùng.

Notebook thực hiện nạp dữ liệu, kiểm tra schema, khảo sát chất lượng ban đầu,
làm sạch đến bước 5.4 và xuất audit. Notebook không thực hiện gắn nhãn, chia tập
dữ liệu hoặc huấn luyện mô hình.


## 0. Thiết lập package cho kernel hiện tại

Cell kế tiếp dùng `%pip` để cài package đúng vào kernel đang chạy. Chạy cell này trước khi chạy các cell import.

In [6]:
# Cài package vào đúng Python kernel hiện tại. Chỉ cần chạy một lần mỗi môi trường.
%pip install pandas regex


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path
import json
import re

import pandas as pd
try:
    import regex as unicode_regex
except ImportError:
    unicode_regex = None

from IPython.display import display


In [8]:
# Đọc cấu hình chung của pipeline

CONFIG_PATHS = [
    Path('Preprocessing/config/preprocessing_config.json'),
    Path('../config/preprocessing_config.json'),
    Path('config/preprocessing_config.json'),
]
CONFIG_PATH = next((path for path in CONFIG_PATHS if path.exists()), None)
if CONFIG_PATH is None:
    raise FileNotFoundError('Không tìm thấy config/preprocessing_config.json.')

with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PREPROCESSING_CONFIG = json.load(config_file)

MIN_COMMENT_WORDS = PREPROCESSING_CONFIG['min_comment_words']
MAX_COMMENT_WORDS = PREPROCESSING_CONFIG['max_comment_words']
RANDOM_SEED = PREPROCESSING_CONFIG['random_seed']

print(f'Đọc cấu hình từ: {CONFIG_PATH.resolve()}')
print(f'Ngưỡng độ dài: {MIN_COMMENT_WORDS}–{MAX_COMMENT_WORDS} từ')


Đọc cấu hình từ: C:\Users\Admin\Desktop\DS108\Preprocessing\config\preprocessing_config.json
Ngưỡng độ dài: 2–200 từ


In [9]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', None)


In [10]:
CANDIDATE_PATHS = [
    Path('Preprocessing/data/raw/raw.csv'),
    Path('../Preprocessing/data/raw/raw.csv'),
    Path('C:/Users/Admin/Desktop/DS108/Preprocessing/data/raw/raw.csv')
]
DATA_PATH = next((path for path in CANDIDATE_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Không tìm thấy raw.csv. Hãy chạy notebook từ DS108 hoặc Preprocessing.'
    )
print(f'Đọc dữ liệu từ: {DATA_PATH.resolve()}')


Đọc dữ liệu từ: C:\Users\Admin\Desktop\DS108\Preprocessing\data\raw\raw.csv


In [11]:
REQUIRED_COLUMNS = {'row_id', 'platform', 'title', 'comment'}
print(f'Schema bắt buộc: {sorted(REQUIRED_COLUMNS)}')


Schema bắt buộc: ['comment', 'platform', 'row_id', 'title']


## 3. Dữ liệu đầu vào

### 3.1. Nguồn dữ liệu

Dữ liệu có thể đến từ Facebook, TikTok, YouTube, Threads và VOZ. Mỗi nguồn
có thể có tên trường, định dạng thời gian và mức độ đầy đủ khác nhau.

### 3.2. Cấu trúc dùng chung

| Trường | Ý nghĩa |
|---|---|
| `row_id` | Mã thứ tự của bản ghi trong dữ liệu hợp nhất |
| `platform` | Nền tảng nguồn |
| `title` | Tiêu đề hoặc nội dung chính của bài đăng |
| `comment` | Nội dung bình luận cần xử lý |

Snapshot raw hiện chỉ có bốn trường; không được suy diễn `post_id`,
`comment_id` hoặc thời gian từ `row_id`. Các metadata nguồn nếu cần cho
khử trùng lặp phải được giữ ở tầng canonical, không tạo giả trong raw.

In [12]:
raw_df = pd.read_csv(
    DATA_PATH,
    encoding='utf-8-sig',
    dtype='string',
    keep_default_na=False,
)
raw_df = raw_df.copy(deep=True)
print(f'Đã đọc {len(raw_df):,} bản ghi.')


Đã đọc 63,697 bản ghi.


In [13]:
missing_columns = REQUIRED_COLUMNS - set(raw_df.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

extra_columns = sorted(set(raw_df.columns) - REQUIRED_COLUMNS)
print(f'Cột ngoài schema raw: {extra_columns or "không có"}')


Cột ngoài schema raw: không có


In [14]:
for column in ['platform', 'title', 'comment']:
    raw_df[column] = raw_df[column].astype('string').fillna('')

df = raw_df.copy(deep=True)
print('Đã chuẩn hóa kiểu dữ liệu text; raw_df vẫn được giữ riêng.')


Đã chuẩn hóa kiểu dữ liệu text; raw_df vẫn được giữ riêng.


In [15]:
schema_report = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing': raw_df.isna().sum(),
})
schema_report['empty_or_whitespace'] = [
    int(raw_df[column].str.strip().eq('').sum())
    if column in raw_df.columns else None
    for column in raw_df.columns
]


In [16]:
display(raw_df.head(3))
display(schema_report)


,row_id,platform,title,comment
0,1,facebook,"Cá nhân Th_ể thấy fan Sen Vàng bất lịch sự, nghèo lại còn vô duyên. Ykr ạ","Cá nhân Th_ể thấy fan Sen Vàng bất lịch sự, nghèo lại còn vô duyên. Ykr ạ"
1,2,facebook,"Cá nhân Th_ể thấy fan Sen Vàng bất lịch sự, nghèo lại còn vô duyên. Ykr ạ","Fan vô văn hóa thật sự, nghèo còn md"
2,3,facebook,"nếu em không tute thì hãy nhận mức án tuhinh nha em, cảm ơn Pháp Luật VN đã lấy lại công bằng cho Bác gái","nếu em không tute thì hãy nhận mức án tuhinh nha em, cảm ơn Pháp Luật VN đã lấy lại công bằng cho Bác gái"


,dtype,missing,empty_or_whitespace
row_id,string,0,0
platform,string,0,0
title,string,0,0
comment,string,0,415


## 4. Khảo sát chất lượng dữ liệu ban đầu

Trước khi làm sạch, cần thống kê và quan sát:

- Số lượng bản ghi và số lượng bình luận có nội dung.
- Số lượng bản ghi theo từng nền tảng.
- Tỷ lệ comment rỗng hoặc chỉ chứa khoảng trắng.
- Số lượng nội dung hệ thống, HTML hoặc lỗi mã hóa.
- Số lượng URL, email, số điện thoại và tên người được tag.
- Số lượng emoji và các trường hợp bình luận chỉ gồm emoji.
- Độ dài ngắn nhất, dài nhất và phân bố độ dài bình luận.

Kết quả khảo sát là cơ sở để lựa chọn quy tắc xử lý ở các bước tiếp theo.
Các số liệu dưới đây chỉ là thống kê ứng viên; chưa tự động xóa URL, PII,
spam hoặc comment ngắn.

In [17]:
comment_text = raw_df['comment'].fillna('').astype('string')
title_text = raw_df['title'].fillna('').astype('string')


In [18]:
SYSTEM_MESSAGE_RE = re.compile(
    r'^\s*(?:comment\s+(?:không|khong)\s+(?:khả dụng|kha dung)|'
    r'comment unavailable|nội dung\s+(?:không|khong)\s+(?:khả dụng|kha dung))\s*$',
    re.IGNORECASE,
)
HTML_TAG_RE = re.compile(r'</?[A-Za-z][^>]*>')
URL_RE = re.compile(r'https?://\S+|www\.\S+|(?<![@\w])(?:[a-z0-9-]+\.)+(?:com|vn|net|org|io|co)(?:/\S*)?', re.IGNORECASE)
EMAIL_RE = re.compile(r'\b[^\s@]+@[^\s@]+\.[^\s@]+\b', re.IGNORECASE)
PHONE_RE = re.compile(r'(?<!\d)(?:\+?84|0)(?:[\s().-]*\d){8,10}(?!\d)')
MENTION_RE = re.compile(r'(?<![\w.])@[\w][\w.-]*', re.UNICODE)


In [19]:
EMOJI_BASE_RE = re.compile(
    r'[\u00a9\u00ae\u203c\u2049\u2122\u2139'
    r'\u2190-\u21ff\u2300-\u23ff\u25aa-\u27bf'
    r'\u2934-\u2935\u2b00-\u2bff\u3030\u303d\u3297\u3299'
    r'\U0001f000-\U0001faff]'
)
EMOJI_MODIFIER_RE = re.compile(r'[\U0001F3FB-\U0001F3FF]')
KEYCAP_RE = re.compile(r'[0-9#*]\ufe0f?\u20e3')
EMOJI_COMPONENTS_RE = re.compile(r'[\ufe0e\ufe0f\u200d\u20e3]')
EMOJI_CLUSTER_RE = (
    unicode_regex.compile(
        r'(?:\p{Extended_Pictographic}(?:\p{Emoji_Modifier}|\uFE0F|\uFE0E)?'
        r'(?:\u200D(?:\p{Extended_Pictographic}|\p{Emoji_Presentation})'
        r'(?:\p{Emoji_Modifier}|\uFE0F|\uFE0E)?)*'
        r'|\p{Regional_Indicator}{2}|[0-9#*]\uFE0F?\u20E3)'
    )
    if unicode_regex is not None
    else None
)


In [20]:
def emoji_tokens(value):
    text = str(value)
    if EMOJI_CLUSTER_RE is not None:
        return EMOJI_CLUSTER_RE.findall(text)
    text_without_modifiers = EMOJI_MODIFIER_RE.sub('', text)
    return EMOJI_BASE_RE.findall(text_without_modifiers) + KEYCAP_RE.findall(text)


In [21]:
def is_emoji_only(value):
    text = str(value).strip()
    if not text or not emoji_tokens(text):
        return False
    residual = KEYCAP_RE.sub('', text)
    residual = EMOJI_BASE_RE.sub('', residual)
    residual = EMOJI_MODIFIER_RE.sub('', residual)
    residual = EMOJI_COMPONENTS_RE.sub('', residual)
    return not residual.strip()


In [22]:
feature_masks = {
    'empty_comment': comment_text.str.strip().eq(''),
    'system_message_candidate': comment_text.map(
        lambda value: bool(SYSTEM_MESSAGE_RE.fullmatch(str(value)))
    ).astype('boolean'),
    'html_tag_candidate': comment_text.map(
        lambda value: bool(HTML_TAG_RE.search(str(value)))
    ).astype('boolean'),
    'encoding_replacement_character': comment_text.str.contains('\ufffd', regex=False),
    'url_candidate': comment_text.map(
        lambda value: bool(URL_RE.search(str(value)))
    ).astype('boolean'),
    'email_candidate': comment_text.map(
        lambda value: bool(EMAIL_RE.search(str(value)))
    ).astype('boolean'),
    'phone_candidate': comment_text.map(
        lambda value: bool(PHONE_RE.search(str(value)))
    ).astype('boolean'),
    'mention_candidate': comment_text.map(
        lambda value: bool(MENTION_RE.search(str(value)))
    ).astype('boolean'),
    'emoji_or_symbol_only': comment_text.map(is_emoji_only).astype('boolean'),
}


In [23]:
emoji_occurrences = comment_text.map(lambda value: len(emoji_tokens(value)))


In [24]:
comment_chars = comment_text.str.strip().str.len()
comment_words = comment_text.str.strip().str.split().str.len()


In [25]:
quality_summary = pd.DataFrame({
    'metric': [
        'total_records', 'comments_with_content', 'empty_comments',
        'rows_with_any_candidate_error', 'rows_with_url_candidate',
        'rows_with_email_candidate', 'rows_with_phone_candidate',
        'rows_with_mention_candidate', 'rows_with_emoji_or_symbol_only',
        'rows_with_any_emoji', 'emoji_occurrences',
    ],
    'value': [
        len(raw_df), int((~feature_masks['empty_comment']).sum()),
        int(feature_masks['empty_comment'].sum()),
        int((feature_masks['system_message_candidate']
             | feature_masks['html_tag_candidate']
             | feature_masks['encoding_replacement_character']).sum()),
        int(feature_masks['url_candidate'].sum()),
        int(feature_masks['email_candidate'].sum()),
        int(feature_masks['phone_candidate'].sum()),
        int(feature_masks['mention_candidate'].sum()),
        int(feature_masks['emoji_or_symbol_only'].sum()),
        int(emoji_occurrences.gt(0).sum()),
        int(emoji_occurrences.sum()),
    ],
})


In [26]:
display(quality_summary)
print('Các cờ trên là ứng viên để review; chưa phải quyết định xóa dữ liệu.')


,metric,value
0,total_records,63697
1,comments_with_content,63282
2,empty_comments,415
3,rows_with_any_candidate_error,1
4,rows_with_url_candidate,677
5,rows_with_email_candidate,3
6,rows_with_phone_candidate,161
7,rows_with_mention_candidate,2213
8,rows_with_emoji_or_symbol_only,489
9,rows_with_any_emoji,7703


Các cờ trên là ứng viên để review; chưa phải quyết định xóa dữ liệu.


In [27]:
platform_summary = (
    raw_df.assign(platform=raw_df['platform'].str.strip().replace('', 'unknown'))
    .groupby('platform', dropna=False)
    .agg(
        records=('comment', 'size'),
        comments_with_content=('comment', lambda s: int(s.str.strip().ne('').sum())),
    )
    .sort_values('records', ascending=False)
)
platform_summary['share'] = platform_summary['records'] / len(raw_df)


In [28]:
display(platform_summary)
print('Cột share được lưu dưới dạng tỷ lệ số; không phụ thuộc jinja2.')


,records,comments_with_content,share
platform,,,
youtube,21891,21889,0.343674
threads,11347,11347,0.178140
tiktok,10991,10991,0.172551
facebook,10074,9888,0.158155
voz,9394,9167,0.147479


Cột share được lưu dưới dạng tỷ lệ số; không phụ thuộc jinja2.


In [29]:
length_summary = pd.DataFrame({
    'field': ['comment'],
    'min_chars': [int(comment_chars.min())],
    'mean_chars': [float(comment_chars.mean())],
    'median_chars': [float(comment_chars.median())],
    'p95_chars': [float(comment_chars.quantile(.95))],
    'max_chars': [int(comment_chars.max())],
    'mean_words': [float(comment_words.mean())],
})


In [30]:
display(length_summary)


,field,min_chars,mean_chars,median_chars,p95_chars,max_chars,mean_words
0,comment,0,94.722028,38.0,312.0,47959,21.748591


## 5. Quy trình tiền xử lý chính

Pipeline dự kiến:

`Dữ liệu đầu vào` → `Kiểm tra lỗi` → `Chuẩn hóa cấu trúc` → `Loại thông tin
không cần thiết` → `Loại spam/quảng cáo` → `Chuẩn hóa văn bản` → `Bảo toàn
emoji và ngữ nghĩa` → `Kiểm tra trùng lặp` → `Kiểm tra chất lượng sau xử lý`

Mỗi bước cần có phần mô tả mục tiêu, quy tắc áp dụng và ví dụ trước–sau.
Trong notebook này, phần code được triển khai đến hết bước 5.4; khử trùng lặp nằm ở notebook 04.

In [31]:
pipeline_steps = pd.DataFrame({
    'step': ['1', '2', '3', '4', '5', '5.1', '5.2', '5.3', '5.4'],
    'name': [
        'Xác định mục tiêu và phạm vi',
        'Giữ raw và xác định nguyên tắc',
        'Nạp raw và kiểm tra schema',
        'Khảo sát chất lượng ban đầu',
        'Thiết lập pipeline tiền xử lý',
        'Kiểm tra comment rỗng và lỗi rõ ràng',
        'Xóa PII, URL, mention và hashtag',
        'Loại comment không hữu ích',
        'Chuẩn hóa Unicode và định dạng',
    ],
    'status': ['đã mô tả', 'đã mô tả', 'đã chạy', 'đã chạy', 'đã thiết lập', 'đã chạy', 'đã chạy', 'đã chạy', 'đã chạy'],
})


In [32]:
display(pipeline_steps)


,step,name,status
0,1,Xác định mục tiêu và phạm vi,đã mô tả
1,2,Giữ raw và xác định nguyên tắc,đã mô tả
2,3,Nạp raw và kiểm tra schema,đã chạy
3,4,Khảo sát chất lượng ban đầu,đã chạy
4,5,Thiết lập pipeline tiền xử lý,đã thiết lập
5,5.1,Kiểm tra comment rỗng và lỗi rõ ràng,đã chạy
6,5.2,"Xóa PII, URL, mention và hashtag",đã chạy
7,5.3,Loại comment không hữu ích,đã chạy
8,5.4,Chuẩn hóa Unicode và định dạng,đã chạy


### 5.1. Kiểm tra dữ liệu rỗng và xóa dòng rỗng

Ở bước này chỉ kiểm tra cột comment có rỗng hoặc chỉ chứa khoảng trắng hay
không. Các comment rỗng sẽ được loại khỏi bản dữ liệu làm việc.

Ví dụ:

| Trước xử lý | Sau xử lý |
|---|---|
| (chỉ khoảng trắng) | Xóa dòng |
| Món này ngon quá | Giữ lại |


In [33]:
# 5.1. Hàm xóa comment rỗng

def remove_empty_comments(data):
    """Trả về dữ liệu có comment và các dòng bị loại vì comment rỗng."""
    working_data = data.copy(deep=True)
    is_empty = working_data['comment'].astype('string').str.strip().eq('')

    removed_rows = working_data.loc[
        is_empty,
        ['row_id', 'platform', 'comment'],
    ].copy()
    removed_rows['remove_reason'] = 'empty_comment'

    cleaned_data = working_data.loc[~is_empty].copy()
    return cleaned_data, removed_rows


In [34]:
# 5.1. Xóa comment rỗng và hiển thị kết quả

comments_without_empty, removed_empty_comments = remove_empty_comments(raw_df)

empty_comment_summary = pd.DataFrame({
    'metric': ['before_empty_check', 'removed_empty_comments', 'after_empty_check'],
    'records': [
        len(raw_df),
        len(removed_empty_comments),
        len(comments_without_empty),
    ],
})

display(empty_comment_summary)
display(removed_empty_comments.head(10))

print('raw_df không bị thay đổi.')
print('comments_without_empty là dữ liệu dùng cho bước tiếp theo.')


,metric,records
0,before_empty_check,63697
1,removed_empty_comments,415
2,after_empty_check,63282


,row_id,platform,comment,remove_reason
12,13,facebook,,empty_comment
383,384,facebook,,empty_comment
711,712,facebook,,empty_comment
1170,1171,facebook,,empty_comment
1172,1173,facebook,,empty_comment
1176,1177,facebook,,empty_comment
1460,1461,facebook,,empty_comment
1847,1848,facebook,,empty_comment
1908,1909,facebook,,empty_comment
1909,1910,facebook,,empty_comment


raw_df không bị thay đổi.
comments_without_empty là dữ liệu dùng cho bước tiếp theo.


### Giải thích kết quả bước 5.1

- before_5_1 = 63.697: tổng số dòng trước khi kiểm tra comment rỗng.
- removed_empty_comment = 415: số dòng có comment rỗng hoặc chỉ chứa khoảng trắng và đã được loại khỏi dữ liệu làm việc.
- after_5_1 = 63.282: số dòng còn lại để chuyển sang bước tiếp theo.

Công thức kiểm tra:

    63.697 - 415 = 63.282

Trong bảng các dòng đã xóa:

- row_id: mã của dòng trong dữ liệu gốc, nên không nhất thiết liên tục.
- platform: nền tảng của bản ghi; các dòng mẫu này đến từ Facebook.
- comment: để trống vì comment không có nội dung.
- invalid_reason = empty_comment: lý do duy nhất khiến dòng bị xóa ở bước 5.1.

Bảng đang hiển thị 10 dòng đầu tiên trong số 415 dòng đã xóa, không phải toàn bộ 415 dòng.


### 5.2. Kiểm tra và xóa PII, hashtag

Kiểm tra email, số điện thoại, đường dẫn, mention và hashtag còn sót lại trong
dữ liệu. Ở bước này chỉ xóa các phần được xác định khỏi bản dữ liệu làm việc;
không xóa cả dòng chỉ vì dòng đó có PII hoặc hashtag.

Ví dụ:

    Gửi qua abc@example.com nhé  →  Gửi qua nhé
    Liên hệ 0900000000 để mua   →  Liên hệ để mua
    Sản phẩm tốt #review #sale  →  Sản phẩm tốt

Raw data vẫn được giữ nguyên để đối chiếu.


In [35]:
# 5.2. Hàm xóa PII và hashtag

HASHTAG_PATTERN = re.compile(r'(?<!\w)#[\w]+', re.UNICODE)

def count_pattern(text, pattern):
    return len(pattern.findall(str(text)))

def remove_sensitive_parts(text):
    """Chỉ xóa URL, email, số điện thoại, mention và hashtag."""
    text = URL_RE.sub(' ', str(text))
    text = EMAIL_RE.sub(' ', text)
    text = PHONE_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)
    text = HASHTAG_PATTERN.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def remove_pii_and_hashtags(data):
    """Tạo dữ liệu sạch và bảng audit riêng để đối chiếu trước/sau."""
    audit = data[['row_id', 'platform', 'title', 'comment']].copy()
    audit['title_before_cleaning'] = audit['title']
    audit['comment_before_cleaning'] = audit['comment']

    text_before_cleaning = (
        audit['title_before_cleaning'].fillna('')
        + ' '
        + audit['comment_before_cleaning'].fillna('')
    )

    for column_name, pattern in [
        ('url_count', URL_RE),
        ('email_count', EMAIL_RE),
        ('phone_count', PHONE_RE),
        ('mention_count', MENTION_RE),
        ('hashtag_count', HASHTAG_PATTERN),
    ]:
        audit[column_name] = text_before_cleaning.map(
            lambda text: count_pattern(text, pattern)
        )

    audit['has_pii_or_hashtag'] = (
        audit[
            ['url_count', 'email_count', 'phone_count',
             'mention_count', 'hashtag_count']
        ]
        .sum(axis=1)
        .gt(0)
    )
    audit['title_after_cleaning'] = audit['title_before_cleaning'].map(
        remove_sensitive_parts
    )
    audit['comment_after_cleaning'] = audit['comment_before_cleaning'].map(
        remove_sensitive_parts
    )

    cleaned_data = data.copy(deep=True)
    cleaned_data['title'] = audit['title_after_cleaning']
    cleaned_data['comment'] = audit['comment_after_cleaning']

    for column_name in [
        'url_count', 'email_count', 'phone_count',
        'mention_count', 'hashtag_count', 'has_pii_or_hashtag',
    ]:
        cleaned_data[column_name] = audit[column_name]

    cleaned_data['comment_empty_after_pii_removal'] = (
        cleaned_data['comment'].str.strip().eq('')
    )
    return cleaned_data, audit


In [36]:
# 5.2. Xóa PII và hashtag khỏi dữ liệu làm việc

comments_without_pii, pii_audit = remove_pii_and_hashtags(comments_without_empty)

pii_summary = pd.DataFrame({
    'metric': [
        'rows_checked',
        'rows_with_pii_or_hashtag',
        'url_occurrences',
        'email_occurrences',
        'phone_occurrences',
        'mention_occurrences',
        'hashtag_occurrences',
        'comments_empty_after_pii_removal',
    ],
    'records': [
        len(comments_without_pii),
        int(comments_without_pii['has_pii_or_hashtag'].sum()),
        int(comments_without_pii['url_count'].sum()),
        int(comments_without_pii['email_count'].sum()),
        int(comments_without_pii['phone_count'].sum()),
        int(comments_without_pii['mention_count'].sum()),
        int(comments_without_pii['hashtag_count'].sum()),
        int(comments_without_pii['comment_empty_after_pii_removal'].sum()),
    ],
})

display(pii_summary)
print('comments_without_pii là dữ liệu dùng cho bước tiếp theo.')


,metric,records
0,rows_checked,63282
1,rows_with_pii_or_hashtag,15012
2,url_occurrences,1066
3,email_occurrences,3
4,phone_occurrences,208
5,mention_occurrences,9786
6,hashtag_occurrences,20198
7,comments_empty_after_pii_removal,83


comments_without_pii là dữ liệu dùng cho bước tiếp theo.


In [37]:
# 5.2. Hiển thị ví dụ PII và hashtag đã xóa

def select_pii_examples(audit, minimum_length=50, sample_size=3):
    """Lấy comment dài, ngẫu nhiên cho từng loại thông tin cần xóa."""
    original_comments = audit['comment_before_cleaning'].fillna('')
    sample_conditions = [
        (
            'Email/URL',
            original_comments.map(
                lambda text: bool(URL_RE.search(str(text)) or EMAIL_RE.search(str(text)))
            ),
        ),
        ('Số điện thoại', original_comments.map(lambda text: bool(PHONE_RE.search(str(text))))),
        ('Hashtag', original_comments.map(lambda text: bool(HASHTAG_PATTERN.search(str(text))))),
        ('Mention', original_comments.map(lambda text: bool(MENTION_RE.search(str(text))))),
    ]
    columns_to_show = [
        'row_id', 'platform',
        'comment_before_cleaning', 'comment_after_cleaning',
        'url_count', 'email_count', 'phone_count',
        'mention_count', 'hashtag_count',
    ]

    example_frames = []
    for example_type, condition in sample_conditions:
        candidates = audit.loc[
            condition & audit['comment_after_cleaning'].str.len().ge(minimum_length),
            columns_to_show,
        ]
        if len(candidates) < sample_size:
            raise ValueError(f'Không đủ {sample_size} ví dụ cho nhóm: {example_type}')

        examples = candidates.sample(n=sample_size, random_state=RANDOM_SEED).copy()
        examples.insert(0, 'example_type', example_type)
        example_frames.append(examples)

    return pd.concat(example_frames, ignore_index=True)

pii_examples = select_pii_examples(pii_audit)
display(pii_examples)


,example_type,row_id,platform,comment_before_cleaning,comment_after_cleaning,url_count,email_count,phone_count,mention_count,hashtag_count
0,Email/URL,19919,threads,"Tham khảo zocker thử nha s.shopee.vn/40fFc… s.shopee.vn [ Đặt Tăng 1 size ] Giày Đá Bóng Zocker Inspire Pro Gen 2| Đá Thật Chân, Form Bền Bỉ, Bám Sân Tốt (Tặng Tất zocker ) | Shopee Việt Nam","Tham khảo zocker thử nha [ Đặt Tăng 1 size ] Giày Đá Bóng Zocker Inspire Pro Gen 2| Đá Thật Chân, Form Bền Bỉ, Bám Sân Tốt (Tặng Tất zocker ) | Shopee Việt Nam",2,0,0,0,0
1,Email/URL,40444,voz,"Chắc vozer hóng hớt ở đâu nghe chưa rõ , nhưng đầu éo có kiến thức để phân tích nên mới thành tam sao thất bản vậy đấy. Cấp đông là để diệt ký sinh trùng (giun sán) chứ không phải vi khuẩn:\nCách đầu bếp nhà hàng cao cấp xử lý sán trong cá sống\nQuá trình đông lạnh cá trên tàu có thể diệt trừ bớt ký sinh trùng, nhưng không đảm bảo giết chết toàn bộ giun sán.\nvnexpress.net","Chắc vozer hóng hớt ở đâu nghe chưa rõ , nhưng đầu éo có kiến thức để phân tích nên mới thành tam sao thất bản vậy đấy. Cấp đông là để diệt ký sinh trùng (giun sán) chứ không phải vi khuẩn: Cách đầu bếp nhà hàng cao cấp xử lý sán trong cá sống Quá trình đông lạnh cá trên tàu có thể diệt trừ bớt ký sinh trùng, nhưng không đảm bảo giết chết toàn bộ giun sán.",1,0,0,0,0
2,Email/URL,12305,threads,"Bài đang viral nên tui ké, có ai trade truyện khônggg threads.com/@zupp… Post not available","Bài đang viral nên tui ké, có ai trade truyện khônggg Post not available",1,0,0,1,0
3,Số điện thoại,12603,threads,"đi du lịch cần thơ , đi cồn sơn đi ạ có đi ib zalo tui nhen ☎️📞 0907646523","đi du lịch cần thơ , đi cồn sơn đi ạ có đi ib zalo tui nhen ☎️📞",0,0,1,0,0
4,Số điện thoại,15942,threads,"mình có tour đi miền tây nề, bạn quan tâm liên hệ mình tư vấn choaaaa, có gì mình gửi bạn chương trìh tour cho bạn tham khảo lun hen SDT/ZALO: 0794993096","mình có tour đi miền tây nề, bạn quan tâm liên hệ mình tư vấn choaaaa, có gì mình gửi bạn chương trìh tour cho bạn tham khảo lun hen SDT/ZALO:",0,0,1,0,0
5,Số điện thoại,827,facebook,"Dương Hải Sơn – Bạn Uống Tôi Lái kính chúc anh chị một ngày cuối tuần vui vẻ và ấm áp bên người thân và bạn bè! Nếu cần tài xế lái xe hộ, anh/chị hãy liên hệ bên em nhé! Phục vụ 24/7 – Nhanh chóng – An toàn – Tận tâm. Hotline: 0364 333 999 | 0774 333 999 (2/8)","Dương Hải Sơn – Bạn Uống Tôi Lái kính chúc anh chị một ngày cuối tuần vui vẻ và ấm áp bên người thân và bạn bè! Nếu cần tài xế lái xe hộ, anh/chị hãy liên hệ bên em nhé! Phục vụ 24/7 – Nhanh chóng – An toàn – Tận tâm. Hotline: | /8)",0,0,2,0,0
6,Hashtag,4708,facebook,Vào mùa tăng cân các bác ạ. Mới đầu mùa đã sương sương 2kg roài\n#banhbeotapgym #fithub #tapluyen,Vào mùa tăng cân các bác ạ. Mới đầu mùa đã sương sương 2kg roài,0,0,0,0,3
7,Hashtag,3912,facebook,"[LỊCH THI ĐẤU ASEAN HYUNDAI CUP 2026] - BẢNG A\nVòng quay của ASEAN Hyundai Cup 2026 sẽ tiếp tục với những màn so tài nảy lửa tại bảng A diễn ra vào tối nay\nSau ngày ra quân ấn tượng với chiến thắng ngược dòng trước Campuchia, Singapore sẽ trở về sân nhà để chạm trán Timor Leste - đội thua tới 0-7 trước tuyển Việt Nam. Với đẳng cấp hơn hẳn đối thủ, cộng thêm tinh thần lên cao, sẽ chẳng có gì bất ngờ nếu đội bóng quốc đảo Sư Tử tiếp tục giành chiến thắng để mở rộng cánh cửa bước vào bán kết\nỞ trận đấu diễn ra sau đó, Indonesia sau khi được nghỉ ngơi trong ngày ra quân sẽ có lần đầu tiên xuất trận tại ASEAN Hyundai Cup 2026 bằng màn chạm trán Campuchia. Lợi thế sân nhà trở thành cơ sở để đội bóng xứ vạn đảo hướng tới 3 điểm trọn vẹn trong cuộc đối đầu này\nỞ lượt trận diễn ra vào ngày 27/7, tuyển Việt Nam sẽ được nghỉ ngơi trước khi trở lại bằng trận đấu gặp Singapore vào ngày 31/7\n#FPTPlay #FPTPlaySports #ASEANHyundaiCup2026 #TrieutraitimMotchienthang","[LỊCH THI ĐẤU ASEAN HYUNDAI CUP 2026] - BẢNG A Vòng quay của ASEAN Hyundai Cup 2026 sẽ tiếp tục với những màn so tài nảy lửa tại bảng A diễn ra vào tối nay Sau ngày ra quân ấn tượng với chiến thắng ngược dòng trước Campuchia, Singapore sẽ trở

### 5.3. Loại bỏ quảng cáo và nội dung không hữu ích

Các bình luận được loại khi thuộc một trong các trường hợp: quảng cáo rõ ràng,
nội dung hệ thống, chuỗi ký tự vô nghĩa, dưới 2 từ hoặc trên 200 từ.

Số từ được tính bằng các cụm ký tự tách bởi khoảng trắng; emoji đơn lẻ như `😂`
hoặc `✅` được tính là 1 từ và bị loại.


In [38]:
# 5.3. Hàm nhận biết comment cần loại bỏ

AD_KEYWORD_PATTERN = re.compile(
    r'\b(?:mua\s*hàng|bán\s*hàng|đặt\s*hàng|order|chốt\s*đơn|'
    r'giá\s*rẻ|giảm\s*giá|khuyến\s*mãi|freeship|ship|'
    r'tuyển\s*dụng|tuyển\s*cộng\s*tác\s*viên)\b',
    re.IGNORECASE,
)
REPEATED_CHARACTER_PATTERN = re.compile(r'^\s*([a-zA-Z0-9])\1{7,}\s*$')

def contains_contact_information(text):
    return bool(
        PHONE_RE.search(str(text))
        or EMAIL_RE.search(str(text))
        or URL_RE.search(str(text))
        or MENTION_RE.search(str(text))
    )

def join_removal_reasons(row):
    reasons = []
    if row['is_too_short']:
        reasons.append('under_2_words')
    if row['is_too_long']:
        reasons.append('over_200_words')
    if row['is_clear_ad']:
        reasons.append('clear_advertising')
    if row['is_system_message']:
        reasons.append('system_message')
    if row['is_meaningless_text']:
        reasons.append('meaningless_repeated_text')
    return ', '.join(reasons)

def identify_unhelpful_comments(data, audit):
    """Gắn cờ comment theo quy tắc độ dài, quảng cáo, hệ thống và vô nghĩa."""
    review_data = data.copy(deep=True)
    original_comments = audit.loc[
        review_data.index, 'comment_before_cleaning'
    ].fillna('').astype('string')

    review_data['comment_word_count'] = (
        review_data['comment'].fillna('').astype('string').str.strip()
        .str.split().str.len().fillna(0).astype(int)
    )
    review_data['is_too_short'] = review_data['comment_word_count'].lt(MIN_COMMENT_WORDS)
    review_data['is_too_long'] = review_data['comment_word_count'].gt(MAX_COMMENT_WORDS)
    review_data['is_clear_ad'] = original_comments.map(
        lambda text: bool(AD_KEYWORD_PATTERN.search(str(text)))
        and contains_contact_information(text)
    )
    review_data['is_system_message'] = review_data['comment'].map(
        lambda text: bool(SYSTEM_MESSAGE_RE.fullmatch(str(text)))
    )
    review_data['is_meaningless_text'] = review_data['comment'].map(
        lambda text: bool(REPEATED_CHARACTER_PATTERN.fullmatch(str(text)))
    )
    review_data['removal_reasons'] = review_data.apply(
        join_removal_reasons,
        axis=1,
    )
    return review_data


In [39]:
# 5.3. Loại bỏ comment không hữu ích

content_review = identify_unhelpful_comments(comments_without_pii, pii_audit)

removed_comments = content_review.loc[
    content_review['removal_reasons'].ne('')
].copy()
filtered_comments = content_review.loc[
    content_review['removal_reasons'].eq('')
].copy()

content_filter_summary = pd.DataFrame({
    'metric': [
        'before_content_filter',
        'flagged_under_2_words',
        'flagged_over_200_words',
        'flagged_clear_advertising',
        'flagged_system_message',
        'flagged_meaningless_text',
        'removed_rows_total',
        'after_content_filter',
    ],
    'records': [
        len(content_review),
        int(content_review['is_too_short'].sum()),
        int(content_review['is_too_long'].sum()),
        int(content_review['is_clear_ad'].sum()),
        int(content_review['is_system_message'].sum()),
        int(content_review['is_meaningless_text'].sum()),
        len(removed_comments),
        len(filtered_comments),
    ],
})

display(content_filter_summary)
display(
    removed_comments[
        ['row_id', 'platform', 'comment', 'comment_word_count', 'removal_reasons']
    ].head(10)
)

print('filtered_comments là dữ liệu dùng cho bước tiếp theo.')


,metric,records
0,before_content_filter,63282
1,flagged_under_2_words,7892
2,flagged_over_200_words,584
3,flagged_clear_advertising,32
4,flagged_system_message,0
5,flagged_meaningless_text,4
6,removed_rows_total,8500
7,after_content_filter,54782


,row_id,platform,comment,comment_word_count,removal_reasons
65,66,facebook,"Nhìn lại vụ án gây chấn động: Một cô hoa hậu bị kết án không trung thực. 1. Miss Grand 2023 Lê Hoàng Phương 'xâm phạm nghiêm trọng' nghĩa vụ đại sứ thương hiệu Theo tòa, hợp đồng giữa Công ty Sen Vàng và Bệnh viện thẩm mỹ Nam An là không rõ ràng, nếu các bên không có thỏa thuận sửa đổi, bổ sung thì tiếp tục thực hiện theo hướng có lợi cho bệnh viện. Riêng đại sứ thương hiệu Lê Hoàng Phương xâm phạm nghiêm trọng đến quyền lợi của bệnh viện. Ngày 13.6, TAND Q.Bình Thạnh (TP.HCM) đã tuyên án vụ kiện ""tranh chấp hợp đồng quảng bá thương hiệu - 149"" giữa nguyên đơn Công ty CP Bệnh viện phẫu thuật thẩm mỹ Nam An (gọi tắt Bệnh viện thẩm mỹ Nam An) và bị đơn Công ty CP quảng cáo thương mại Sen Vàng (gọi tắt Công ty Sen Vàng). Vụ kiện liên quan đến đại sứ thương hiệu Lê Hoàng Phương. Miss Grand Việt Nam 2023 Lê Hoàng Phương tham gia phiên tòa với tư cách người liên quan, ông Chiêm Quốc Thái (Giám đốc Bệnh viện thẩm mỹ Việt Mỹ) là người làm chứng. 2. Công ty Sen Vàng chịu trách nhiệm khi hoa hậu, á hậu có vi phạm Theo HĐXX, với hợp đồng 149, nghĩa vụ của bị đơn, là Ban tổ chức cuộc thi Miss Grand Việt Nam 2023 phải đảm bảo hoa hậu trở thành đại sứ thương hiệu cho nguyên đơn, nếu bị đơn thay đổi hoặc hoa hậu không đồng ý thì bị đơn phải chịu trách nhiệm pháp lý. Tòa cũng nêu, theo tập quán trong hoạt động thương mại, đại sứ thương hiệu là người đại diện cho nhãn hàng về hình ảnh, truyền thông và thực hiện các công việc theo thỏa thuận với nhãn hàng, doanh nghiệp để quảng bá cho thương hiệu. Đại sứ thương hiệu thường là người nổi tiếng, sở hữu lượng người hâm mộ tương đối lớn, họ là người hưởng lợi trực tiếp từ việc quảng bá. Hơn nữa, hoa hậu sau đăng quang trở thành đại sứ thương hiệu là cam kết mà Công ty Sen Vàng phải thực hiện với nguyên đơn, đồng thời phải chịu trách nhiệm với bất cứ vi phạm nào của hoa hậu và á hậu khi thực hiện không đúng, không đủ. Tại phiên tòa, các bên thừa nhận bà Lê Hoàng Phương không thông báo cho nguyên đơn và bị đơn về mối quan hệ, sự thỏa thuận sử dụng hình ảnh của bà Lê Hoàng Phương với ông Chiêm Quốc Thái. Theo tòa, cho dù bất cứ lý do gì, hành vi này thể hiện bà Lê Hoàng Phương không trung thực khi cung cấp thông tin cá nhân để trở thành đại sứ thương hiệu cho nguyên đơn. Sự trung thực này mang tính quyết định của nguyên đơn là có giao kết hợp đồng đại sứ thương hiệu.",502,over_200_words
73,74,facebook,?,1,under_2_words
95,96,facebook,"NHÌN LẠI SỰ CỐ GÂY CHẤN ĐỘNG: VỤ SẬP SÂN KHẤU TRƯỚC ĐÊM BÁN KẾT TỪNG KHIẾN CẢ CUỘC THI LAO ĐAO Cách đây một thời gian, dư luận từng rúng động trước sự cố sập sân khấu ngay trước thềm đêm bán kết của một cuộc thi sắc đẹp lớn – một biến cố được xem là “vết đen” khó quên trong khâu tổ chức sự kiện. Theo những gì được ghi nhận vào thời điểm đó, sự cố xảy ra trong buổi tổng duyệt cuối cùng. Khi các thí sinh đang trình diễn catwalk trên sân khấu chính, một tiếng nứt lớn bất ngờ vang lên. Chỉ trong tích tắc, phần sàn phía trước đổ sập hoàn toàn, kéo theo hệ thống đèn và khung kim loại rơi xuống, khiến hiện trường rơi vào hỗn loạn. Hậu quả nghiêm trọng nhất là một thí sinh đã bị kẹt dưới cấu trúc sân khấu và gãy chân, buộc phải nhập viện khẩn cấp. Nhiều thí sinh khác dù may mắn thoát nạn nhưng không tránh khỏi hoảng loạn và ám ảnh tâm lý sau sự cố. Thời điểm đó, hình ảnh sân khấu tan hoang, thiết bị vỡ vụn lan truyền mạnh mẽ trên mạng xã hội, gây nên làn sóng tranh cãi dữ dội. Công chúng đặt ra hàng loạt câu hỏi về tiêu chuẩn an toàn, quy trình kiểm tra và trách nhiệm của ban tổ chức. Sau sự cố, cuộc thi buộc phải tạm hoãn để xử lý hậu quả. Đêm bán kết – vốn được kỳ vọng – bị dời lịch, thậm chí đứng trước nguy cơ hủy bỏ hoàn toàn. Uy tín của cuộc thi cũng bị ảnh hưởng nặng nề, kéo theo phản ứng từ nhà tài trợ và khán giả. Về phía ban tổ chức, sau đó đã có những động thái khắc phục, kiểm tra lại toàn bộ kết cấu sân khấu và cam kết siết chặt quy trình an toàn trong các sự kiện tiếp theo. Tuy nhiên, sự cố 

filtered_comments là dữ liệu dùng cho bước tiếp theo.


### 5.4. Chuẩn hóa Unicode, khoảng trắng và định dạng

Chỉ chuẩn hóa các lỗi kỹ thuật như Unicode không thống nhất, khoảng trắng
thừa, xuống dòng bất thường hoặc HTML do crawler thu thập. 

Ví dụ:

```text
  Sản   phẩm   này   tốt!!!  →  Sản phẩm này tốt!!!
```

In [40]:
# 5.4. Nạp công cụ chuẩn hóa kỹ thuật

import html
import unicodedata


In [41]:
# 5.4. Hàm chuẩn hóa Unicode, HTML và khoảng trắng

def normalize_technical_text(text):
    """Chỉ sửa lỗi kỹ thuật, giữ emoji, dấu câu và cách viết tự nhiên."""
    text = html.unescape(str(text))
    text = HTML_TAG_RE.sub(' ', text)
    text = unicodedata.normalize('NFC', text)
    text = text.replace('﻿', '').replace('​', ' ')
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'[ \u00a0]+', ' ', text)
    text = text.strip()
    text = re.sub(r'^(?:·\s*)+', '', text)
    return text.strip()

def normalize_text_columns(data):
    """Chuẩn hóa title và comment, đồng thời trả về bảng đối chiếu."""
    audit = data[['row_id', 'platform', 'title', 'comment']].copy()
    audit['title_before_normalization'] = audit['title']
    audit['comment_before_normalization'] = audit['comment']

    normalized_data = data.copy(deep=True)
    normalized_data['title'] = normalized_data['title'].map(normalize_technical_text)
    normalized_data['comment'] = normalized_data['comment'].map(normalize_technical_text)

    audit['title_after_normalization'] = normalized_data['title']
    audit['comment_after_normalization'] = normalized_data['comment']
    audit['title_changed'] = (
        audit['title_before_normalization'] != audit['title_after_normalization']
    )
    audit['comment_changed'] = (
        audit['comment_before_normalization'] != audit['comment_after_normalization']
    )

    normalized_data['unicode_or_format_changed'] = (
        audit['title_changed'] | audit['comment_changed']
    )
    return normalized_data, audit


In [42]:
# 5.4. Chuẩn hóa dữ liệu và kiểm tra kết quả

normalized_comments, format_audit = normalize_text_columns(filtered_comments)

text_normalization_summary = pd.DataFrame({
    'metric': [
        'before_text_normalization',
        'titles_changed',
        'comments_changed',
        'rows_changed',
        'after_text_normalization',
    ],
    'records': [
        len(filtered_comments),
        int(format_audit['title_changed'].sum()),
        int(format_audit['comment_changed'].sum()),
        int(normalized_comments['unicode_or_format_changed'].sum()),
        len(normalized_comments),
    ],
})

format_examples = format_audit.loc[
    format_audit['title_changed'] | format_audit['comment_changed'],
    [
        'row_id', 'platform',
        'comment_before_normalization', 'comment_after_normalization',
    ],
].head(10)

display(text_normalization_summary)
display(format_examples)

print('normalized_comments là dữ liệu sạch sau bước 5.4.')


,metric,records
0,before_text_normalization,54782
1,titles_changed,382
2,comments_changed,2584
3,rows_changed,2952
4,after_text_normalization,54782


,row_id,platform,comment_before_normalization,comment_after_normalization
755,756,facebook,Chúc mừng Tú,Chúc mừng Tú
1226,1227,facebook,Quen bài lám rồi,Quen bài lám rồi
2006,2007,facebook,6 tỉ mà lấy ng k có gì là sao dọ. Ảnh làm đã hẻ,6 tỉ mà lấy ng k có gì là sao dọ. Ảnh làm đã hẻ
2137,2138,facebook,Nếu biết WC and chills thì không phát ngôn thế đâu.,Nếu biết WC and chills thì không phát ngôn thế đâu.
2454,2455,facebook,nam mô Adi đa phật,nam mô Adi đa phật
3201,3202,facebook,thằng tốt nhất rạp xiếc xl thì 4 thằng kia sao chịu nổi :)))),thằng tốt nhất rạp xiếc xl thì 4 thằng kia sao chịu nổi :))))
3944,3945,facebook,Thương cảm phục và biết ơn mẹ người mẹ vĩ đại,Thương cảm phục và biết ơn mẹ người mẹ vĩ đại
4063,4064,facebook,"Bão số 2 đi sâu vào đất liền tỉnh Quảng Đông (Trung Quốc), suy yếu dần và không có khả năng ảnh hưởng trực tiếp đến đất liền Việt Nam.","Bão số 2 đi sâu vào đất liền tỉnh Quảng Đông (Trung Quốc), suy yếu dần và không có khả năng ảnh hưởng trực tiếp đến đất liền Việt Nam."
4064,4065,facebook,May quá rồi. Mong cho Vn ít bão gió,May quá rồi. Mong cho Vn ít bão gió
4065,4066,facebook,Cho ké tí mưa đc r,Cho ké tí mưa đc r


normalized_comments là dữ liệu sạch sau bước 5.4.


In [43]:
# 5.4. Viết hoa chữ cái đầu tiên của comment

def capitalize_first_letter(text):
    """Viết hoa chữ cái đầu tiên, giữ nguyên các ký tự còn lại."""
    text = str(text)
    for position, character in enumerate(text):
        if character.isalpha() and character.lower() != character.upper():
            return text[:position] + character.upper() + text[position + 1:]
    return text

capitalized_comments = normalized_comments.copy(deep=True)
original_comments = capitalized_comments['comment'].copy()
capitalized_comments['comment'] = capitalized_comments['comment'].map(
    capitalize_first_letter
)

capitalization_summary = pd.DataFrame({
    'metric': ['comments_checked', 'comments_with_capitalized_start'],
    'records': [
        len(capitalized_comments),
        int(original_comments.ne(capitalized_comments['comment']).sum()),
    ],
})

display(capitalization_summary)


,metric,records
0,comments_checked,54782
1,comments_with_capitalized_start,19024


## 6. Kiểm tra chất lượng sau tiền xử lý

Sau khi hoàn tất các bước làm sạch, cần kiểm tra lại:

- Còn bình luận rỗng hoặc nội dung hệ thống hay không.
- Emoji Unicode có bị thay đổi hoặc mất đi không.
- Tên người được tag và thông tin nhận dạng không cần thiết đã được xử lý
  đúng chưa.
- Các bình luận hợp lệ có bị loại nhầm hay không.
- Số lượng bản ghi trước và sau từng bước thay đổi như thế nào.
- Các trường hợp nghi ngờ có được giữ lại để kiểm tra hay không.

## 7. Kết quả mong đợi

Sau khi hoàn thiện các cell code, notebook sẽ cung cấp một quy trình tiền xử
lý có thể kiểm tra theo từng bước. Kết quả cần bảo đảm dữ liệu có cấu trúc
thống nhất hơn, giảm nội dung lỗi và spam, đồng thời vẫn giữ được emoji, dấu
câu và cách diễn đạt tự nhiên của người dùng.

## 8. Kiểm tra cuối và xuất dữ liệu sau làm sạch

Bước này kiểm tra lại comment rỗng phát sinh sau khi xóa PII, quảng cáo và
chuẩn hóa kỹ thuật. Các bảng audit được lưu riêng trong data/audit.


In [44]:
# Hàm xuất dataset đã làm sạch

def export_cleaned_dataset(data, output_folder, file_name):
    """Xuất row_id và comment bằng UTF-8-SIG để mở đúng tiếng Việt trong Excel."""
    columns_to_export = ['row_id', 'comment']
    output_folder.mkdir(parents=True, exist_ok=True)
    output_path = output_folder / file_name

    data.loc[:, columns_to_export].to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig',
    )
    return output_path


In [45]:
# Kiểm tra cuối, lưu audit và tạo bảng tổng kết pipeline

final_empty_mask = capitalized_comments['comment'].fillna('').astype('string').str.strip().eq('')
final_empty_comments = capitalized_comments.loc[final_empty_mask].copy()
final_cleaned_comments = capitalized_comments.loc[~final_empty_mask].copy()

AUDIT_FOLDER = DATA_PATH.parent.parent / 'audit'
AUDIT_FOLDER.mkdir(parents=True, exist_ok=True)

def save_audit_table(data, file_name):
    data.to_csv(AUDIT_FOLDER / file_name, index=False, encoding='utf-8-sig')

audit_outputs = {
    'removed_empty_comments.csv': removed_empty_comments,
    'pii_audit.csv': pii_audit,
    'content_removal_audit.csv': removed_comments,
    'format_audit.csv': format_audit,
    'final_empty_comments.csv': final_empty_comments,
}
for file_name, audit_data in audit_outputs.items():
    save_audit_table(audit_data, file_name)

pipeline_summary = pd.DataFrame({
    'metric': [
        'raw_records', 'removed_empty_comments', 'after_empty_check',
        'rows_checked_for_pii', 'rows_with_pii_or_hashtag',
        'removed_content_rows', 'after_content_filter',
        'after_final_empty_check', 'removed_after_final_empty_check',
    ],
    'records': [
        len(raw_df), len(removed_empty_comments), len(comments_without_empty),
        len(comments_without_pii), int(comments_without_pii['has_pii_or_hashtag'].sum()),
        len(removed_comments), len(filtered_comments), len(final_cleaned_comments),
        len(final_empty_comments),
    ],
})
save_audit_table(pipeline_summary, 'pipeline_summary.csv')
(AUDIT_FOLDER / 'preprocessing_config_used.json').write_text(
    json.dumps(PREPROCESSING_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(pipeline_summary)
print(f'Đã lưu audit vào: {AUDIT_FOLDER.resolve()}')


,metric,records
0,raw_records,63697
1,removed_empty_comments,415
2,after_empty_check,63282
3,rows_checked_for_pii,63282
4,rows_with_pii_or_hashtag,15012
5,removed_content_rows,8500
6,after_content_filter,54782
7,after_final_empty_check,54782
8,removed_after_final_empty_check,0


Đã lưu audit vào: C:\Users\Admin\Desktop\DS108\Preprocessing\data\audit


In [46]:
# Xuất dữ liệu sau làm sạch để làm đầu vào khử trùng lặp

export_folder = DATA_PATH.parent.parent / 'interim'
exported_dataset_path = export_cleaned_dataset(
    final_cleaned_comments, export_folder, 'comments_cleaned_to_5_4.csv'
)

export_summary = pd.DataFrame({
    'metric': ['rows_exported', 'columns_exported', 'output_path'],
    'value': [len(final_cleaned_comments), 2, str(exported_dataset_path.resolve())],
})
display(export_summary)
print(f'Đã xuất dữ liệu sau làm sạch: {exported_dataset_path.resolve()}')


,metric,value
0,rows_exported,54782
1,columns_exported,2
2,output_path,C:\Users\Admin\Desktop\DS108\Preprocessing\data\interim\comments_cleaned_to_5_4.csv


Đã xuất dữ liệu sau làm sạch: C:\Users\Admin\Desktop\DS108\Preprocessing\data\interim\comments_cleaned_to_5_4.csv
